# 24. Multi-platform calibration: one API, honest status

`qb-compiler` reads device calibration for IBM, Rigetti, IonQ, IQM and Quantinuum through one
registry. This notebook shows how to get a provider for any supported backend, and, more
importantly, **how to tell what you are actually looking at**: live data, a real cached
snapshot, or synthetic values.

That distinction is the point. A calibration number you cannot trace is worse than no number,
because you will believe it.


## Every backend reports its own status

`all_backend_statuses()` answers the question you should ask before trusting any calibration:
where would this data come from, right now, in this environment?


In [1]:
from qb_compiler.calibration.registry import (
    LiveStatus,
    all_backend_statuses,
    get_backend_status,
    get_calibration_provider,
)

statuses = all_backend_statuses()
print(f'{len(statuses)} backends known to the registry')
print()
print(f"{'backend':<18} {'provider':<12} {'live status':<12} {'live deps':<10} {'static'}")
print('-' * 62)
for s in sorted(statuses, key=lambda x: (x.provider, x.backend)):
    print(
        f'{s.backend:<18} {s.provider:<12} {s.live_status.value:<12} '
        f'{str(s.live_deps_available):<10} {s.static_available}'
    )


9 backends known to the registry

backend            provider     live status  live deps  static
--------------------------------------------------------------
ibm_fez            ibm          live         True       True
ibm_marrakesh      ibm          live         True       False
ibm_torino         ibm          live         True       True
ionq_aria          ionq         live-unvalidated True       False
ionq_forte         ionq         live-unvalidated True       False
iqm_emerald        iqm          live-unvalidated True       False
iqm_garnet         iqm          live-unvalidated True       False
quantinuum_h2      quantinuum   live-unvalidated True       False
rigetti_ankaa      rigetti      live-unvalidated True       True


### What the statuses mean

| field | meaning |
|---|---|
| `live_status = live` | the live path is implemented AND validated against the real device |
| `live_status = live-unvalidated` | the live path is implemented but not yet validated end to end |
| `live_deps_available` | the vendor SDK is importable in this environment |
| `static_available` | a **real cached snapshot** ships for this backend; if false, the static
fallback is synthetic |

`live-unvalidated` is deliberately not called `live`. The code path exists and the deps import,
but it has not been proven against hardware, so it is labelled as such rather than overclaimed.


## Getting a provider, without credentials

`prefer_live=False` skips the live path entirely, so this works offline and in CI.


In [2]:
provider = get_calibration_provider('ibm_fez', prefer_live=False)
props = provider.backend_properties

print('backend      :', props.backend)
print('provider     :', props.provider)
print('qubits       :', props.n_qubits)
print('basis gates  :', props.basis_gates)
print('couplings    :', len(props.coupling_map))
print('live status  :', get_backend_status('ibm_fez').live_status.value)


backend      : ibm_fez
provider     : ibm
qubits       : 156
basis gates  : ('sx', 'reset', 'rz', 'delay', 'cz', 'if_else', 'id', 'x', 'measure')
couplings    : 352
live status  : live


## The same call shape across vendors

IBM, Braket, Quantinuum and Azure differ enormously underneath. The registry's job is that your
code does not have to care.


In [3]:
# Real registry names. Asking for a backend the registry does not know is an error, not a
# silent fallback, so these are the exact identifiers.
for backend in ['ibm_fez', 'ibm_torino', 'rigetti_ankaa', 'ionq_aria', 'quantinuum_h2', 'iqm_garnet']:
    bp = get_calibration_provider(backend, prefer_live=False).backend_properties
    st = get_backend_status(backend)
    print(
        f'{backend:<16} {bp.provider:<12} {bp.n_qubits:>4} qubits   '
        f'live={st.live_status.value:<18} real_snapshot={st.static_available}'
    )


ibm_fez          ibm           156 qubits   live=live               real_snapshot=True
ibm_torino       ibm           133 qubits   live=live               real_snapshot=True
rigetti_ankaa    rigetti        84 qubits   live=live-unvalidated   real_snapshot=True
ionq_aria        ionq           25 qubits   live=live-unvalidated   real_snapshot=False
quantinuum_h2    quantinuum     32 qubits   live=live-unvalidated   real_snapshot=False
iqm_garnet       iqm            20 qubits   live=live-unvalidated   real_snapshot=False


## Live access degrades, it does not explode

If you ask for live data and the credentials are missing, the registry falls back to a static
snapshot rather than raising. Your pipeline keeps running; the status tells you what you got.

This is deliberate. A compiler that dies on absent credentials is unusable in CI; one that
silently substitutes synthetic data without saying so is dangerous. The fallback plus the status
flag is the middle path.


In [4]:
prov = get_calibration_provider('ibm_fez', prefer_live=True)
st = get_backend_status('ibm_fez')

print('live status      :', st.live_status.value)
print('live deps present:', st.live_deps_available)
print('static fallback  :', st.static_available)
print('usable properties:', prov.backend_properties is not None)
print()
print('Nothing raised. Read live_status rather than assuming the data is live.')


live status      : live
live deps present: True
static fallback  : True


usable properties: True

Nothing raised. Read live_status rather than assuming the data is live.


## Unknown backends fail loudly

The fallback covers *missing credentials*, not *typos*. An unsupported backend is an error, so a
misspelt name cannot quietly become synthetic data.


In [5]:
from qb_compiler.exceptions import BackendNotSupportedError

try:
    get_calibration_provider('ibm_definitely_not_a_real_device', prefer_live=False)
    print('no error raised')
except BackendNotSupportedError as exc:
    print('BackendNotSupportedError:', exc)


BackendNotSupportedError: Backend 'ibm_definitely_not_a_real_device' is not supported. Available: ibm_fez, ibm_torino, ibm_marrakesh, rigetti_ankaa, ionq_aria, ionq_forte, iqm_garnet, iqm_emerald, quantinuum_h2


## Summary

- one registry call for every supported vendor
- `get_backend_status()` tells you live vs real-cached vs synthetic **before** you trust a number
- missing credentials degrade to static data instead of crashing
- unknown backends raise, so typos cannot masquerade as data

Provenance is the feature here, not the convenience.
